# Lesson 02: Knowledge Representation, Ontologies & Expert Systems

In this Colab notebook, you will:
1. Build a **Backward-Chaining** and **Forward-Chaining** Animal Expert System.
2. Construct and query an **RDF / OWL Ontology** using `rdflib`.
3. Explore open-source **Concept Graph** (`is-a`) taxonomic reasoning.

In [ ]:
!pip install -q rdflib

## Part 1: Backward-Chaining Expert System Shell

In [ ]:
class Content:
    def __init__(self, x):
        self.x = x

class If(Content):
    pass

class AND(Content):
    pass

class OR(Content):
    pass

rules = {
    'mammal': If(OR(['hair', 'gives milk'])),
    'carnivore': If(OR([AND(['sharp teeth', 'claws', 'forward-looking eyes']), 'eats meat'])),
    'ungulate': If(['mammal', OR(['has hooves', 'chews cud'])]),
    'bird': If(OR(['feathers', AND(['flies', 'lays eggs'])])),
    'animal:monkey': If(['mammal', 'carnivore', 'color:red-brown', 'pattern:dark spots']),
    'animal:tiger': If(['mammal', 'carnivore', 'color:red-brown', 'pattern:dark stripes']),
    'animal:giraffe': If(['ungulate', 'long neck', 'long legs', 'pattern:dark spots']),
    'animal:zebra': If(['ungulate', 'pattern:dark stripes']),
    'animal:penguin': If(['bird', 'swims', 'color:black and white', 'cannot fly']),
    'animal:albatross': If(['bird', 'flies well']),
}

class KnowledgeBase:
    def __init__(self, rules, initial_facts=None):
        self.rules = rules
        self.memory = dict(initial_facts or {})

    def get(self, name):
        if ':' in name and name not in self.rules:
            k, v = name.split(':', 1)
            return 'y' if self.memory.get(k) == v else 'n'
        if name in self.memory:
            return self.memory[name]
        for fld, rule in self.rules.items():
            if fld == name or fld.startswith(name + ':'):
                val = 'y' if fld == name else fld.split(':', 1)[1]
                if self.eval(rule) == 'y':
                    self.memory[name] = val
                    return val
        return 'n'

    def eval(self, expr):
        if isinstance(expr, If):
            return self.eval(expr.x)
        if isinstance(expr, (AND, list)):
            items = expr.x if isinstance(expr, AND) else expr
            return 'y' if all(self.eval(item) == 'y' for item in items) else 'n'
        if isinstance(expr, OR):
            return 'y' if any(self.eval(item) == 'y' for item in expr.x) else 'n'
        if isinstance(expr, str):
            return self.get(expr)
        return 'n'

sample_facts = {
    'gives milk': 'y',
    'has hooves': 'y',
    'long neck': 'y',
    'long legs': 'y',
    'pattern': 'dark spots',
}
kb = KnowledgeBase(rules, sample_facts)
print("Inferred Animal:", kb.get('animal'))

## Part 2: Semantic Web RDF Ontology with `rdflib`

Let's build a miniature family ontology using RDF triplets and query grandparents automatically with SPARQL.

In [ ]:
from rdflib import Graph, Namespace, RDF, Literal

EX = Namespace("http://example.org/family/")
g = Graph()

# Add Subject-Predicate-Object triplets
g.add((EX.Alice, EX.parentOf, EX.Bob))
g.add((EX.Bob, EX.parentOf, EX.Charlie))
g.add((EX.Alice, EX.name, Literal("Alice")))
g.add((EX.Bob, EX.name, Literal("Bob")))
g.add((EX.Charlie, EX.name, Literal("Charlie")))

query = """
PREFIX ex: <http://example.org/family/>
SELECT ?grandparentName ?grandchildName
WHERE {
  ?gp ex:parentOf ?p .
  ?p ex:parentOf ?gc .
  ?gp ex:name ?grandparentName .
  ?gc ex:name ?grandchildName .
}
"""

for row in g.query(query):
    print(f"{row.grandparentName} is a grandparent of {row.grandchildName}")

## Part 3: Open-Source Concept Graph Categorization

Instead of proprietary REST APIs, we can score text against an open `is-a` concept taxonomy using weighted probabilities.

In [ ]:
concept_graph = {
    "python": [("programming_language", 0.92), ("snake", 0.41)],
    "pytorch": [("deep_learning_framework", 0.95), ("software_library", 0.88)],
    "tesla": [("electric_vehicle_company", 0.89), ("physicist", 0.72)],
    "retina": [("anatomy", 0.85), ("display_brand", 0.45)],
}

headline = "PyTorch and Python power modern deep learning"
scores = {}
for token in headline.lower().split():
    for category, prob in concept_graph.get(token, []):
        scores[category] = scores.get(category, 0.0) + prob

print("Inferred Topic Scores:", sorted(scores.items(), key=lambda x: -x[1]))